# Experiment 5 — CelebA with our flow model

Same Experiment 2 winning U-Net and Experiment 4 training recipe (Adam, stepped LR, EMA). The dataset changes to aligned CelebA at 64x64 RGB, so `in_channels` is 3. Official train/valid/test splits are used.

In [4]:
import os
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
os.chdir(project_root)

import torch
import wandb

from datasets.celeba import CelebASampler
from models.config import load_config
from models.flow import FlowModel
from training.path import GaussianConditionalProbabilityPath, LinearAlpha, LinearBeta
from training.trainer import FlowTrainer, model_size_b

CONFIG_PATH = "configs/experiment_5.yaml"
cfg = load_config(CONFIG_PATH)
persistent_root = Path(
    os.getenv("DIFFUSION_DATA_ROOT", "/workspace-global/Diffusion-data")
)
data_root = Path("data/raw")
if not data_root.exists():
    data_root = persistent_root / "datasets"
checkpoints_dir = Path("checkpoints") / "experiment_5"
samples_dir = Path("samples") / "experiment_5"
# W&B copies images with chmod; GeeseFS rejects that. Metrics still persist on wandb.ai.
wandb_dir = Path("/workspace/wandb")
for path in (data_root, checkpoints_dir, samples_dir, wandb_dir):
    path.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("Experiment 5 is intended to run on a CUDA GPU")
torch.set_float32_matmul_precision("high")
torch.manual_seed(cfg["data"]["seed"])
print("device", device, torch.cuda.get_device_name(0))
print("image size", cfg["data"]["image_size"], "channels", cfg["unet"]["in_channels"])

device cuda NVIDIA A40
image size 64 channels 3


In [5]:
def make_path(split: str) -> GaussianConditionalProbabilityPath:
    image_size = cfg["data"]["image_size"]
    return GaussianConditionalProbabilityPath(
        p_data=CelebASampler(
            root=str(data_root),
            split=split,
            image_size=image_size,
            seed=cfg["data"]["seed"],
        ),
        p_simple_shape=[3, image_size, image_size],
        alpha=LinearAlpha(),
        beta=LinearBeta(),
    ).to(device)


train_path = make_path("train")
val_path = make_path("val")
model = FlowModel.from_config(CONFIG_PATH).to(device)
trainer = FlowTrainer(path=train_path, model=model, val_path=val_path)
print(f"train images: {len(train_path.p_data.images):,}")
print(f"val images: {len(val_path.p_data.images):,}")
print(f"model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"model size: {model_size_b(model) / 1024**2:.2f} MiB")

train images: 162,770
val images: 19,867
model parameters: 29,832,835
model size: 113.80 MiB


In [6]:
wandb.login()
train_cfg = cfg["training"]
sampling_cfg = cfg["sampling"]
existing = sorted(checkpoints_dir.glob("*.pt"))
resume_ckpt = next((path for path in existing if not path.name.endswith("_best.pt")), None)

with wandb.init(
    project="celeba-flow-matching",
    dir=str(wandb_dir),
    name="experiment-5-celeba",
    id=resume_ckpt.stem if resume_ckpt else None,
    resume="must" if resume_ckpt else None,
    tags=["experiment-5", "celeba", "step-lr", "ema"],
    config=cfg,
) as run:
    checkpoint_path = checkpoints_dir / f"{run.id}.pt"
    best_checkpoint_path = checkpoints_dir / f"{run.id}_best.pt"
    history = trainer.train(
        num_steps=train_cfg["num_steps"],
        device=device,
        lr=train_cfg["learning_rate"],
        optimizer_name=train_cfg["optimizer"],
        weight_decay=train_cfg["weight_decay"],
        max_grad_norm=train_cfg["max_grad_norm"],
        lr_milestones=train_cfg["lr_milestones"],
        lr_gamma=train_cfg["lr_gamma"],
        ema_decay=train_cfg["ema_decay"],
        batch_size=cfg["data"]["batch_size"],
        ckpt_path=checkpoint_path,
        best_ckpt_path=best_checkpoint_path,
        checkpoint_every=train_cfg["checkpoint_every"],
        val_every=train_cfg["val_every"],
        val_batches=train_cfg["val_batches"],
        plot_every=train_cfg["plot_every"],
        n_plot_images=sampling_cfg["n_plot_images"],
        n_plot_steps=sampling_cfg["n_plot_steps"],
        samples_dir=samples_dir,
        show_plots=False,
        wandb_run=run,
        resume_from=resume_ckpt,
    )
    best_train_loss = history["train"].min().item()
    best_val_loss = history["val"].min().item()
    run.summary["loss/train_best"] = best_train_loss
    run.summary["loss/val_best"] = best_val_loss
    run.summary["checkpoint"] = str(checkpoint_path)
    run.summary["best_checkpoint"] = str(best_checkpoint_path)
    run.summary["training_steps"] = train_cfg["num_steps"]

best_state = torch.load(best_checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(best_state["ema_model"] or best_state["model"])
model.eval()
print("best training loss", best_train_loss)
print("best EMA validation loss", best_val_loss)
print("best checkpoint step", best_state["step"])
print("best checkpoint", best_checkpoint_path)

Training model with size: 113.803 MiB
resumed checkpoints/experiment_5/ybap4pux.pt at step 1000


Step 20000, train: 0.129, val: 0.109: 100%|██████████| 19000/19000 [54:17<00:00,  5.83it/s]  


saved checkpoints/experiment_5/ybap4pux.pt


learning_rate,████████████████████████▃▃▃▃▃▃▁▁▁▁▁▁▁▁▁▁
loss/train,▇▄▇▅▆▅█▄▅▆▂▅▆█▃▇▃▃█▆▅▇▃▅▄▂▃▃▂▃▁▃▄▅▃▆▆▂▄▇
loss/train_best,█▆▅▅▅▅▅▅▄▄▄▄▄▄▄▄▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁
loss/val,█▆▅▅▃▃▂▂▂▂▂▁▂▁▂▁▁▁▁▂▁▂▂▁▂▁▂▁▂▁▁▂▁▁▂▁▂▁▂▁
loss/val_best,█▅▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_checkpoint,checkpoints/experime...
checkpoint,checkpoints/experime...
learning_rate,1e-05
loss/train,0.12946
loss/train_best,0.06862
loss/val,0.10915


best training loss 0.06861560046672821
best EMA validation loss 0.10332544893026352
best checkpoint step 12000
best checkpoint checkpoints/experiment_5/ybap4pux_best.pt


In [7]:
import torch.nn.functional as F
from torchvision.utils import save_image

from sampling.ode import EulerSimulator, FlowODE

image_size = cfg["data"]["image_size"]
num_samples = cfg["sampling"]["num_samples"]
ode_steps = cfg["sampling"]["ode_steps"]
generator = torch.Generator(device=device).manual_seed(cfg["data"]["seed"])
noise = torch.randn(
    num_samples, 3, image_size, image_size, generator=generator, device=device
)
ts = torch.linspace(0, 1, ode_steps + 1, device=device).expand(num_samples, -1)
samples = EulerSimulator(FlowODE(model)).simulate(noise, ts).clamp(-1, 1).cpu()

generation_dir = samples_dir / "final"
individual_dir = generation_dir / "individual"
individual_dir.mkdir(parents=True, exist_ok=True)
torch.save(samples, generation_dir / "samples.pt")
display_samples = F.interpolate(samples, size=(256, 256), mode="bilinear", align_corners=False)
save_image(
    display_samples,
    generation_dir / "grid_5x5.png",
    nrow=5,
    normalize=True,
    value_range=(-1, 1),
    padding=4,
    pad_value=1,
)
for index, sample in enumerate(display_samples):
    save_image(
        sample,
        individual_dir / f"sample_{index + 1:02d}.png",
        normalize=True,
        value_range=(-1, 1),
    )

print("grid", generation_dir / "grid_5x5.png")

100%|██████████| 400/400 [00:14<00:00, 28.51it/s]


grid samples/experiment_5/final/grid_5x5.png
